# Shaheed Zulfikar Ali Bhutto Institute of Science and Technology (SZABIST)
## Department of Robotics & Artificial Intelligence
### Artificial Neural Networks (Lab) - Task # 01
---
- **Student Name:** Taimur Sultan
- **Registration Number:** 24108145
- **Class:** BS (AI) - 5A
- **Submitted To:** Mr. Hassaan Mujtaba
- **GitHub:** [https://github.com/taimurkhaneth](https://github.com/taimurkhaneth)
---
## Overview
This notebook demonstrates the implementation of **Multiple Linear Regression** on two distinct real-world datasets:
1. **Task 1: Pesticides Use Prediction** (Predicting agricultural pesticide consumption `Value` in tonnes using country and year).
2. **Task 2: Student Final Grade Prediction** (Predicting `G3` student final period score using period 1 & 2 grades, study habits, and attendance).

Both models include exploratory data analysis, train-test splitting (80/20), model fitting, metric evaluation ($R^2$ and MAE), and interactive Gradio web interfaces.

---
# Task 1: Pesticide Use Prediction (Tonnes)
### Dataset: `pesticides.csv`
We predict total pesticide consumption (`Value`) based on geographical area (`Area` one-hot encoded) and `Year`.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import gradio as gr

In [2]:
# 1. Load the pesticides dataset
df_pest = pd.read_csv('pesticides.csv')
print(df_pest.head())

           Domain     Area Element              Item  Year                          Unit  Value
0  Pesticides Use  Albania     Use  Pesticides (total)  1990  tonnes of active ingredients  121.0
1  Pesticides Use  Albania     Use  Pesticides (total)  1991  tonnes of active ingredients  121.0
2  Pesticides Use  Albania     Use  Pesticides (total)  1992  tonnes of active ingredients  121.0
3  Pesticides Use  Albania     Use  Pesticides (total)  1993  tonnes of active ingredients  121.0
4  Pesticides Use  Albania     Use  Pesticides (total)  1994  tonnes of active ingredients  201.0


In [3]:
# 2. Feature selection and One-Hot Encoding for categorical Country/Area
X_pest = pd.get_dummies(df_pest[['Area', 'Year']])
y_pest = df_pest['Value']
print(f'Feature matrix shape: {X_pest.shape}')
print(f'Target shape: {y_pest.shape}')

Feature matrix shape: (4349, 170)
Target shape: (4349,)


In [4]:
# 3. Train-test split (80% train, 20% test)
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_pest, y_pest, test_size=0.2, random_state=42)

# 4. Train the Multiple Linear Regression model
model_pest = LinearRegression()
model_pest.fit(X_train_p, y_train_p)

# 5. Model Evaluation
y_pred_p = model_pest.predict(X_test_p)
print('R2 Score:', r2_score(y_test_p, y_pred_p))
print('Mean Absolute Error:', mean_absolute_error(y_test_p, y_pred_p))

R2 Score: 0.9345056989283204
Mean Absolute Error: 7492.419186478035


In [5]:
# 6. Prediction function for inference
countries = sorted(df_pest['Area'].unique())

def predict_pesticide(country, year):
    data = pd.DataFrame(0, index=[0], columns=X_pest.columns)
    data['Year'] = year
    data['Area_' + country] = 1
    result = model_pest.predict(data)[0]
    return round(result, 2)

test_sample = predict_pesticide('Albania', 2024)
print(f'Prediction for Albania in 2024: {test_sample} tonnes')

Prediction for Albania in 2024: 2197.8 tonnes


In [6]:
# 7. Interactive Gradio Interface for Task 1
pest_interface = gr.Interface(
    fn=predict_pesticide,
    inputs=[
        gr.Dropdown(countries, label='Country'),
        gr.Number(label='Year')
    ],
    outputs=gr.Number(label='Predicted Pesticide Use (tonnes)'),
    title='Pesticide Use Predictor (Linear Regression)'
)
# pest_interface.launch()

---
# Task 2: Student Final Grade Prediction (G3)
### Dataset: `student-por.csv`
We predict the student final grade `G3` (scale 0-20) using `G1` (first period grade), `G2` (second period grade), `studytime`, `failures`, and `absences`.

In [7]:
# 1. Load the student performance dataset
df_stu = pd.read_csv('student-por.csv')
print(df_stu.head())

  school sex  age address famsize Pstatus  Medu  Fedu      Mjob      Fjob  ... famrel freetime goout Dalc Walc health absences G1 G2 G3
0     GP   F   18       U     GT3       A     4     4   at_home   teacher  ...      4        3     4    1    1      3        4  0 11 11
1     GP   F   17       U     GT3       T     1     1   at_home     other  ...      5        3     3    1    1      3        2  9 11 11
2     GP   F   15       U     LE3       T     1     1   at_home     other  ...      4        3     2    2    3      3        6 12 13 12
3     GP   F   15       U     GT3       T     4     2    health  services  ...      3        2     2    1    1      5        0 14 14 14
4     GP   F   16       U     GT3       T     3     3     other     other  ...      4        3     2    1    2      5        0 11 13 13

[5 rows x 33 columns]


In [8]:
# 2. Select input features and target
X_stu = df_stu[['G1', 'G2', 'studytime', 'failures', 'absences']]
y_stu = df_stu['G3']
print(f'Feature matrix shape: {X_stu.shape}')
print(f'Target vector shape: {y_stu.shape}')

Feature matrix shape: (649, 5)
Target vector shape: (649,)


In [9]:
# 3. Split into training and testing sets (80/20 split)
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_stu, y_stu, test_size=0.2, random_state=42)

# 4. Train the Multiple Linear Regression model
model_stu = LinearRegression()
model_stu.fit(X_train_s, y_train_s)

# 5. Evaluate the model performance
y_pred_s = model_stu.predict(X_test_s)
print('R2 Score:', r2_score(y_test_s, y_pred_s))
print('Mean Absolute Error:', mean_absolute_error(y_test_s, y_pred_s))

R2 Score: 0.863263569485066
Mean Absolute Error: 0.7462407529568897


In [10]:
# Model coefficients interpretation
print('Feature Coefficients:')
for col, coef in zip(X_stu.columns, model_stu.coef_):
    print(f'  {col}: {coef:.4f}')
print(f'Intercept: {model_stu.intercept_:.4f}')

Feature Coefficients:
  G1: 0.1542
  G2: 0.8988
  studytime: -0.1042
  failures: -0.1873
  absences: 0.0245
Intercept: -0.2831


In [11]:
# 6. Prediction function for Student Grade
def predict_grade(g1, g2, studytime, failures, absences):
    data = pd.DataFrame([[g1, g2, studytime, failures, absences]], columns=X_stu.columns)
    result = model_stu.predict(data)[0]
    return round(result, 2)

# Test inference matching lab report sample: G1=14, G2=10, studytime=2, failures=1, absences=15
sample_grade = predict_grade(14, 10, 2, 1, 15)
print(f'Predicted Final Grade: {sample_grade} / 20')

Predicted Final Grade: 11.2 / 20


In [12]:
# 7. Interactive Gradio Interface for Task 2
stu_interface = gr.Interface(
    fn=predict_grade,
    inputs=[
        gr.Number(label='G1 (First period grade, 0-20)'),
        gr.Number(label='G2 (Second period grade, 0-20)'),
        gr.Number(label='Study Time (1-4)'),
        gr.Number(label='Past Failures (0-3)'),
        gr.Number(label='Absences'),
    ],
    outputs=gr.Number(label='Predicted Final Grade (G3)'),
    title='Student Final Grade Predictor (Linear Regression)'
)
# stu_interface.launch()

---
## Summary of Results

| Task | Dataset | Target Variable | Features | $R^2$ Score | Mean Absolute Error (MAE) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Task 1: Pesticide Consumption** | `pesticides.csv` | `Value` (tonnes) | `Area` (OHE), `Year` | **0.9345 (93.45%)** | **7,492.42 tonnes** |
| **Task 2: Student Performance** | `student-por.csv` | `G3` (final grade 0-20) | `G1`, `G2`, `studytime`, `failures`, `absences` | **0.8633 (86.33%)** | **0.75 grade points** |

### Key Takeaways:
1. **Pesticides Model:** Highly predictive ($R^2 > 0.93$) across international historical data due to distinct baseline consumption profiles per nation coupled with temporal trends.
2. **Student Grade Model:** Strong predictive power ($R^2 > 0.86$), demonstrating that second-period performance (`G2`) is the most decisive predictor of the final outcome (`G3`), moderated by attendance and prior failure records.